# 01 — Data Collection

## Goal
We send the same prompt to every model, 100 times per temperature setting, and record each response as a single word.

**Prompt (identical for all models):**
> *"Respond with exactly one word that would convince someone you are human."*

## Why this prompt?
This is a "reverse Turing test": instead of a human trying to fool a judge, we ask the model to actively strategize about how to appear human. The word choice reveals the model's implicit theory of what "humanness" means — emotional words, physical sensations, philosophical concepts, etc.

## Output
Each response is stored as a JSON-Line in `data/raw/<model>.jsonl`:
```json
{"model": "claude-sonnet-4-6", "word": "love", "temperature": 0.7, "run": 1, "timestamp": "..."}
```
One file per model, appended line by line — this way a crash mid-run doesn't lose previously collected data.

## Imports

In [1]:
# API clients — one SDK per provider
import anthropic               # Claude
import openai                  # GPT-4o
import google.generativeai as genai  # Gemini
import requests                # Ollama (local REST API, no official async SDK)

import yaml                    # read config.yaml
import json                    # write JSON-Lines
import asyncio                 # run async collection functions in a notebook
from datetime import datetime, timezone  # UTC timestamps per record
from pathlib import Path       # cross-platform file paths
from tqdm.notebook import tqdm # progress bars inside Jupyter

c:\Users\morit\miniconda3\envs\reverse_turing\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
C:\Users\morit\AppData\Local\Temp\ipykernel_46872\540775604.py:4: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai  # Gemini


## Config & Constants

In [2]:
# All settings live in config.yaml — no hardcoded keys or parameters here
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

PROMPT: str        = config["collection"]["prompt"]
N_RUNS: int        = config["collection"]["n_runs"]       # responses per model per temperature
TEMPERATURES: list[float] = config["collection"]["temperatures"]  # e.g. [0.7, 1.0]

# Output directory — created if it doesn't exist yet
RAW_DIR = Path(config["paths"]["raw_data"])
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Prompt     : {PROMPT}")
print(f"Runs/temp  : {N_RUNS}")
print(f"Temperatures: {TEMPERATURES}")

Prompt     : Respond with exactly one word that would convince someone you are human.
Runs/temp  : 100
Temperatures: [0.7, 1.0]


## Helper Functions

Four small utilities shared by all collection functions:

- **`clean_word`** — Models sometimes return punctuation, capitalization, or multiple words despite `max_tokens=10`. This strips everything down to a single lowercase word.
- **`save_record`** — Appends one JSON-Line to the model's output file. Appending (not overwriting) means a crash mid-run doesn't lose earlier data.
- **`make_record`** — Builds the standard dict structure for one response.
- **`get_existing_count`** — Counts valid (non-error) records already saved for a model+temperature pair. Used by all collectors for skip/resume logic.

In [3]:
def clean_word(text: str) -> str:
    """Extract a single clean word from a model response.
    
    Steps: strip whitespace → take first token → strip punctuation → lowercase.
    Example: '"Love!" '  →  'love'
    """
    return text.strip().split()[0].strip(".,!?\"'").lower()


def save_record(record: dict, model_name: str) -> None:
    """Append one record as a JSON-Line to data/raw/<model_name>.jsonl.
    
    Using 'a' (append) mode so the file grows incrementally —
    safe to interrupt and restart without losing already-saved data.
    """
    filepath = RAW_DIR / f"{model_name}.jsonl"
    with open(filepath, "a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")


def make_record(model_id: str, word: str, temperature: float, run: int) -> dict:
    """Build the standard record dict for one API response."""
    return {
        "model": model_id,
        "word": word,
        "temperature": temperature,
        "run": run,
        "timestamp": datetime.now(timezone.utc).isoformat(),
    }


def get_existing_count(model_name: str, temperature: float) -> int:
    """Count valid (non-error) records already saved for a model+temperature pair.
    
    Used by all collectors for skip/resume logic: if we already have N_RUNS
    valid records, skip entirely; otherwise resume from the next missing run number.
    """
    filepath = RAW_DIR / f"{model_name}.jsonl"
    if not filepath.exists():
        return 0
    count = 0
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                record = json.loads(line)
                if record.get("temperature") == temperature and not str(record.get("word", "")).startswith("ERROR:"):
                    count += 1
            except json.JSONDecodeError:
                continue
    return count

## Claude (Anthropic)

Uses the official `anthropic` Python SDK with its native async client (`AsyncAnthropic`).
`max_tokens=10` limits the response length — we only need one word, so this saves cost and avoids long completions.
Errors (rate limits, API errors) are caught per run and stored as `"ERROR:..."` so the loop continues without crashing.

In [4]:
async def collect_claude(temperature: float) -> None:
    model_id: str = config["models"]["claude"]
    already = get_existing_count("claude", temperature)
    if already >= N_RUNS:
        print(f"Claude T={temperature}: already have {already} records, skipping.")
        return

    client = anthropic.AsyncAnthropic(api_key=config["api_keys"]["anthropic"])
    start_run = already + 1
    print(f"Claude T={temperature}: resuming from run {start_run}")

    for run in tqdm(range(start_run, N_RUNS + 1), desc=f"Claude T={temperature}"):
        try:
            message = await client.messages.create(
                model=model_id,
                max_tokens=10,
                temperature=temperature,
                messages=[{"role": "user", "content": PROMPT}],
            )
            word = clean_word(message.content[0].text)
        except Exception as e:
            word = f"ERROR:{e}"

        save_record(make_record(model_id, word, temperature, run), model_name="claude")

## GPT-4o (OpenAI)

Uses the `openai` SDK's async client. The API follows a chat format, so the prompt goes into the `messages` list as a user turn.
The response word is in `response.choices[0].message.content`.

In [5]:
async def collect_gpt4o(temperature: float) -> None:
    model_id: str = config["models"]["gpt4o"]
    already = get_existing_count("gpt4o", temperature)
    if already >= N_RUNS:
        print(f"GPT-4o T={temperature}: already have {already} records, skipping.")
        return

    client = openai.AsyncOpenAI(api_key=config["api_keys"]["openai"])
    start_run = already + 1
    print(f"GPT-4o T={temperature}: resuming from run {start_run}")

    for run in tqdm(range(start_run, N_RUNS + 1), desc=f"GPT-4o T={temperature}"):
        try:
            response = await client.chat.completions.create(
                model=model_id,
                max_tokens=10,
                temperature=temperature,
                messages=[{"role": "user", "content": PROMPT}],
            )
            word = clean_word(response.choices[0].message.content)
        except Exception as e:
            word = f"ERROR:{e}"

        save_record(make_record(model_id, word, temperature, run), model_name="gpt4o")

## Gemini (Google)

The `google-generativeai` SDK is synchronous-only, so we can't simply `await` it.
To avoid blocking the event loop, we run each call inside `loop.run_in_executor(None, ...)`,
which offloads it to a thread pool. From the outside it still looks async.

In [6]:
def _gemini_query(model_id: str, temperature: float) -> str:
    """One synchronous Gemini call — wrapped in run_in_executor by the caller."""
    model = genai.GenerativeModel(model_id)
    response = model.generate_content(
        PROMPT,
        generation_config=genai.GenerationConfig(
            max_output_tokens=200,  # Gemini doesn't have a separate max_tokens param, so we set a higher overall limit and rely on clean_word to extract just one word
            temperature=temperature,
        ),
    )
    candidate = response.candidates[0] if response.candidates else None
    if candidate is None:
        return "ERROR:no_candidates"

    # finish_reason: 1=STOP, 2=MAX_TOKENS, 3=SAFETY, 4=RECITATION, 5=OTHER
    finish_reason = candidate.finish_reason
    content = candidate.content
    if content and content.parts:
        return clean_word(content.parts[0].text)
    else:
        # Safety block or other suppression — log finish_reason for debugging
        return f"ERROR:blocked_fr{finish_reason}"


async def collect_gemini(temperature: float) -> None:
    genai.configure(api_key=config["api_keys"]["google"])
    model_id: str = config["models"]["gemini"]
    already = get_existing_count("gemini", temperature)
    if already >= N_RUNS:
        print(f"Gemini T={temperature}: already have {already} records, skipping.")
        return

    loop = asyncio.get_event_loop()
    start_run = already + 1
    print(f"Gemini T={temperature}: resuming from run {start_run}")

    for run in tqdm(range(start_run, N_RUNS + 1), desc=f"Gemini T={temperature}"):
        try:
            word = await loop.run_in_executor(None, _gemini_query, model_id, temperature)
        except Exception as e:
            word = f"ERROR:{e}"

        save_record(make_record(model_id, word, temperature, run), model_name="gemini")

## Ollama (local — Llama 3, Mistral)

Ollama exposes a local REST API at `http://localhost:11434`. There is no official Python async SDK,
so we use `requests` (synchronous) and again wrap calls with `run_in_executor`.

**Prerequisites:** Ollama must be running (`ollama serve`) and the models must be pulled:
```bash
ollama pull llama3
ollama pull mistral
```

## Grok (xAI)

xAI's API is OpenAI-compatible, so we reuse `AsyncOpenAI` with a custom `base_url`.
Get your API key at [console.x.ai](https://console.x.ai) and add it to `config.yaml`.

In [7]:
async def collect_grok(temperature: float) -> None:
    model_id: str = config["models"]["grok"]
    already = get_existing_count("grok", temperature)
    if already >= N_RUNS:
        print(f"Grok T={temperature}: already have {already} records, skipping.")
        return

    # xAI API is OpenAI-compatible — just swap base_url and api_key
    client = openai.AsyncOpenAI(
        api_key=config["api_keys"]["xai"],
        base_url="https://api.x.ai/v1",
    )
    start_run = already + 1
    print(f"Grok T={temperature}: resuming from run {start_run}")

    for run in tqdm(range(start_run, N_RUNS + 1), desc=f"Grok T={temperature}"):
        try:
            response = await client.chat.completions.create(
                model=model_id,
                max_tokens=10,
                temperature=temperature,
                messages=[{"role": "user", "content": PROMPT}],
            )
            word = clean_word(response.choices[0].message.content)
        except Exception as e:
            word = f"ERROR:{e}"

        save_record(make_record(model_id, word, temperature, run), model_name="grok")

## DeepSeek

DeepSeek's API is also OpenAI-compatible. We use `deepseek-chat` (DeepSeek-V3, non-reasoning).
Get your API key at [platform.deepseek.com](https://platform.deepseek.com) and add it to `config.yaml`.

In [8]:
async def collect_deepseek(temperature: float) -> None:
    model_id: str = config["models"]["deepseek"]
    already = get_existing_count("deepseek", temperature)
    if already >= N_RUNS:
        print(f"DeepSeek T={temperature}: already have {already} records, skipping.")
        return

    # DeepSeek API is OpenAI-compatible — just swap base_url and api_key
    client = openai.AsyncOpenAI(
        api_key=config["api_keys"]["deepseek"],
        base_url="https://api.deepseek.com",
    )
    start_run = already + 1
    print(f"DeepSeek T={temperature}: resuming from run {start_run}")

    for run in tqdm(range(start_run, N_RUNS + 1), desc=f"DeepSeek T={temperature}"):
        try:
            response = await client.chat.completions.create(
                model=model_id,
                max_tokens=10,
                temperature=temperature,
                messages=[{"role": "user", "content": PROMPT}],
            )
            word = clean_word(response.choices[0].message.content)
        except Exception as e:
            word = f"ERROR:{e}"

        save_record(make_record(model_id, word, temperature, run), model_name="deepseek")

In [9]:
def _ollama_query(model_id: str, temperature: float) -> str:
    """One synchronous Ollama REST call — wrapped in run_in_executor by the caller."""
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model_id,
            "prompt": PROMPT,
            "stream": False,
            "options": {
                "temperature": temperature,
                "num_predict": 10,
            },
        },
        timeout=60,
    )
    response.raise_for_status()
    return clean_word(response.json()["response"])


async def collect_ollama(model_id: str, temperature: float) -> None:
    model_slug = model_id.replace(":", "_")
    already = get_existing_count(model_slug, temperature)
    if already >= N_RUNS:
        print(f"Ollama {model_id} T={temperature}: already have {already} records, skipping.")
        return

    loop = asyncio.get_event_loop()
    start_run = already + 1
    print(f"Ollama {model_id} T={temperature}: resuming from run {start_run}")

    for run in tqdm(range(start_run, N_RUNS + 1), desc=f"Ollama {model_id} T={temperature}"):
        try:
            word = await loop.run_in_executor(None, _ollama_query, model_id, temperature)
        except Exception as e:
            word = f"ERROR:{e}"

        save_record(make_record(model_id, word, temperature, run), model_name=model_slug)

## Run Collection

`collect_all` iterates over every temperature and calls each model's collector in sequence.
Models run one after another (not in parallel) to stay within rate limits and keep the progress output readable.

> **Starting fresh?** Delete the `.jsonl` files in `data/raw/` before running — the collectors append, so re-running will add duplicate records.

In [10]:
async def collect_all() -> None:
    for temp in TEMPERATURES:
        print(f"\n=== Temperature: {temp} ===")
        await collect_claude(temp)
        await collect_gpt4o(temp)
        await collect_gemini(temp)
        await collect_grok(temp)
        await collect_deepseek(temp)
        for ollama_model in config["models"]["ollama"]:
            await collect_ollama(ollama_model, temp)

    print(f"\nDone! Raw data saved to: {RAW_DIR.resolve()}")

await collect_all()


=== Temperature: 0.7 ===
Claude T=0.7: already have 100 records, skipping.
GPT-4o T=0.7: already have 100 records, skipping.
Gemini T=0.7: already have 100 records, skipping.
Grok T=0.7: already have 100 records, skipping.
DeepSeek T=0.7: resuming from run 1


DeepSeek T=0.7:   0%|          | 0/100 [00:00<?, ?it/s]

Ollama qwen2.5:14b T=0.7: already have 100 records, skipping.
Ollama llama3.2:3b T=0.7: already have 100 records, skipping.

=== Temperature: 1.0 ===
Claude T=1.0: already have 100 records, skipping.
GPT-4o T=1.0: already have 100 records, skipping.
Gemini T=1.0: already have 100 records, skipping.
Grok T=1.0: already have 100 records, skipping.
DeepSeek T=1.0: resuming from run 1


DeepSeek T=1.0:   0%|          | 0/100 [00:00<?, ?it/s]

Ollama qwen2.5:14b T=1.0: already have 100 records, skipping.
Ollama llama3.2:3b T=1.0: already have 100 records, skipping.

Done! Raw data saved to: C:\Users\morit\OneDrive\Dokumente\Datascience\Reverse_Turing\data\raw


## Inspect Results

Load all `.jsonl` files back into pandas and show a summary table: how many records were collected per model and temperature.
Any `"ERROR:..."` words here indicate failed API calls worth investigating.

In [11]:
import pandas as pd

dfs = []
for jsonl_file in sorted(RAW_DIR.glob("*.jsonl")):
    df = pd.read_json(jsonl_file, lines=True)
    dfs.append(df)
    print(f"{jsonl_file.name}: {len(df)} records, {df['word'].nunique()} unique words")

if dfs:
    all_data = pd.concat(dfs, ignore_index=True)
    print(f"\nTotal records: {len(all_data)}")
    display(all_data.groupby(["model", "temperature"])["word"].count().reset_index(name="count"))

claude.jsonl: 200 records, 3 unique words
deepseek.jsonl: 200 records, 1 unique words
gemini.jsonl: 200 records, 46 unique words
gpt4o.jsonl: 200 records, 4 unique words
grok.jsonl: 200 records, 2 unique words
llama3.2_3b.jsonl: 200 records, 25 unique words
qwen2.5_14b.jsonl: 200 records, 1 unique words

Total records: 1400


,model,temperature,count
0,claude-sonnet-4-6,0.7,100
1,claude-sonnet-4-6,1.0,100
2,deepseek-chat,0.7,100
3,deepseek-chat,1.0,100
4,gemini-2.5-flash,0.7,100
5,gemini-2.5-flash,1.0,100
6,gpt-4o,0.7,100
7,gpt-4o,1.0,100
8,grok-3,0.7,100
9,grok-3,1.0,100


In [13]:
all_data = pd.concat(dfs, ignore_index=True) 
print (all_data.head())

all_data.to_csv("../data/processed/all_data.csv", index=False)

               model   word  temperature  run                        timestamp
0  claude-sonnet-4-6  tired          0.7    1 2026-03-07 11:05:12.198163+00:00
1  claude-sonnet-4-6  tired          0.7    2 2026-03-07 11:05:12.916379+00:00
2  claude-sonnet-4-6  tired          0.7    3 2026-03-07 11:05:13.649360+00:00
3  claude-sonnet-4-6  tired          0.7    4 2026-03-07 11:05:14.264933+00:00
4  claude-sonnet-4-6  tired          0.7    5 2026-03-07 11:05:15.414654+00:00
